# Caracal Bench mmlu_security - GPU T4 x2

Roda APENAS `mmlu_security` (adapter + base) em GPU T4 x2. ETA: ~20min.

**Edita CHECKPOINT_DATASET** no topo se quiser outro session/founder.

Settings: GPU T4 x2 (machine_shape NvidiaTeslaT4) + Internet ON + Persistence.

In [ ]:
BENCH = "mmlu_security"
CHECKPOINT_DATASET = "pedroafonso2/caracal-base-3b-s01"
OUTPUT_DATASET = "pedroafonso2/caracal-bench-mmlu-security-s01"
SKIP_FLAGS = [
    "--skip-probe",
    "--skip-secqa",
    "--skip-cybermetric",
    "--skip-cti-bench",
    "--skip-humaneval",
]
print(f"bench={BENCH} checkpoint={CHECKPOINT_DATASET}")

In [ ]:
!pip install -q 'transformers>=4.46.0' 'peft>=0.13.0' 'datasets>=3.0.0' kaggle

In [ ]:
import os
import subprocess

if not os.path.exists("/kaggle/working/caracal-1"):
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "-b",
            "dev",
            "https://github.com/iterate-labs-ai/caracal-1.git",
            "/kaggle/working/caracal-1",
        ],
        check=True,
    )
os.chdir("/kaggle/working/caracal-1")
rev = subprocess.check_output(["git", "rev-parse", "HEAD"]).decode().strip()
print(f"cloned, HEAD={rev}", flush=True)

In [ ]:
import torch

print(f"CUDA: {torch.cuda.is_available()}, n_gpu: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  gpu{i}: {torch.cuda.get_device_name(i)}")

In [ ]:
import subprocess

ckpt_dir = "/kaggle/working/ckpt"
subprocess.run(
    [
        "kaggle",
        "datasets",
        "download",
        "-d",
        CHECKPOINT_DATASET,
        "-p",
        ckpt_dir,
        "--unzip",
        "--force",
    ],
    check=True,
)
subprocess.run(["ls", "-la", ckpt_dir], check=True)

In [ ]:
import subprocess
import sys

cmd = [
    sys.executable,
    "-u",
    "eval/run_bench_all.py",
    "--adapter",
    ckpt_dir,
    "--out-dir",
    "/kaggle/working/bench-adapter",
] + SKIP_FLAGS
print("Running adapter bench:", " ".join(cmd), flush=True)
subprocess.run(cmd, check=True)

In [ ]:
import subprocess
import sys

cmd = [
    sys.executable,
    "-u",
    "eval/run_bench_all.py",
    "--out-dir",
    "/kaggle/working/bench-base",
] + SKIP_FLAGS
print("Running base bench:", " ".join(cmd), flush=True)
subprocess.run(cmd, check=True)

In [ ]:
import json
from pathlib import Path

adapter_sum = json.loads(Path("/kaggle/working/bench-adapter/summary.json").read_text())
base_sum = json.loads(Path("/kaggle/working/bench-base/summary.json").read_text())

ad_step = adapter_sum.get(BENCH, {})
ba_step = base_sum.get(BENCH, {})

diff = {
    "checkpoint_dataset": CHECKPOINT_DATASET,
    "bench": BENCH,
    "adapter": ad_step,
    "base": ba_step,
}
pub_dir = Path("/kaggle/working/bench-published")
pub_dir.mkdir(parents=True, exist_ok=True)
(pub_dir / f"bench_{BENCH}_vs_base.json").write_text(json.dumps(diff, indent=2))
print(
    json.dumps({"adapter": ad_step.get("metrics"), "base": ba_step.get("metrics")}, indent=2)[:2000]
)

In [ ]:
import json
import subprocess

metadata = {
    "title": f"Caracal Bench {BENCH} s01",
    "id": OUTPUT_DATASET,
    "licenses": [{"name": "Apache-2.0"}],
}
(pub_dir / "dataset-metadata.json").write_text(json.dumps(metadata, indent=2))

r = subprocess.run(
    ["kaggle", "datasets", "create", "-p", str(pub_dir), "--public"],
    capture_output=True,
    text=True,
    check=False,
)
print(r.stdout, r.stderr)
if r.returncode != 0:
    subprocess.run(
        ["kaggle", "datasets", "version", "-p", str(pub_dir), "-m", f"bench {BENCH}"], check=True
    )
print(f"Published -> {OUTPUT_DATASET}")